# PoliMillionaire - BM25 + BERT + LLM + math tools (Colab)

Pipeline Colab con:

- BM25 multi-index su SimpleWiki + KELM;
- reranking BERT cross-encoder sui candidati;
- tool matematici deterministici via SymPy prima del generativo;
- LLM istruito piccolo come fallback RAG.

Dai documenti locali (`modelli_possibili.md` e slide LLM) la scelta piu prudente per una T4 gratuita e un LLM istruito 1B-2B con prompt corto. Il default e `Qwen/Qwen2.5-1.5B-Instruct`; se il runtime va in OOM, passare a `Qwen/Qwen2.5-0.5B-Instruct`.

Prima di eseguire il notebook in Colab, la cartella Drive deve contenere almeno:

```text
/content/drive/MyDrive/NLP_polimi_26/
  api_client/NLP_assignment_api_client/
  src/retrieval_quiz_runner.py
  src/agentic_tools.py
  indexes/simplewiki_160w_bm25.joblib
  indexes/kelm_500k_bm25.joblib
  logs/
```

In [ ]:
# Se necessario, abilita GPU in Colab: Runtime > Change runtime type > T4 GPU.
!pip -q install sentence-transformers bm25s joblib numpy scikit-learn sympy requests transformers accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

# Percorso Drive gia usato nei notebook Colab del progetto. Non modificarlo qui se la tua repo e in questa posizione.
PROJECT_ROOT = Path('/content/drive/MyDrive/NLP_polimi_26')

CLIENT_PARENT = PROJECT_ROOT / 'api_client' / 'NLP_assignment_api_client'
SRC_DIR = PROJECT_ROOT / 'src'

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print('Project root:', PROJECT_ROOT)
print('Client path exists:', CLIENT_PARENT.exists())
print('Source path exists:', SRC_DIR.exists())

In [ ]:
import json
import re
import time
from getpass import getpass

import numpy as np
import torch
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import (
    load_retrieval_index,
    retrieve,
    compact_snippet,
    RetrievalDecision,
    run_all_competitions,
    summarize,
    write_logs,
)
from agentic_tools import choose_with_agentic_tools

API_URL = os.getenv('POLIMILLIONAIRE_API_URL', 'http://131.175.15.22:51111/')
USERNAME = os.getenv('POLIMILLIONAIRE_USERNAME') or input('Username: ').strip()
PASSWORD = os.getenv('POLIMILLIONAIRE_PASSWORD') or getpass('Password: ')

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} ({user.role})')

In [ ]:
WIKI_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'simplewiki_160w_bm25.joblib'
KELM_INDEX_PATH = PROJECT_ROOT / 'indexes' / 'kelm_500k_bm25.joblib'

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
kelm_index = load_retrieval_index(KELM_INDEX_PATH)
indexes = [wiki_index, kelm_index]

print('Loaded SimpleWiki:', len(wiki_index['docs']))
print('Loaded KELM:', len(kelm_index['docs']))

In [ ]:
from sentence_transformers import CrossEncoder

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('BERT device:', DEVICE)

bert_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2', max_length=512, device=DEVICE)

LOCAL_STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'in', 'is', 'it',
    'of', 'on', 'or', 'the', 'to', 'was', 'were', 'with', 'what', 'which', 'who',
    'when', 'where'
}

def significant_terms(text):
    return [
        t for t in re.findall(r'[a-z0-9][a-z0-9_+-]*', str(text).lower())
        if t not in LOCAL_STOPWORDS and len(t) > 2
    ]

def passage_for_bert(doc, query, max_chars=1400):
    title = str(doc.get('title') or '')
    text = re.sub(r'\s+', ' ', str(doc.get('text') or '')).strip()
    if len(text) <= max_chars:
        return f'{title}: {text}' if title else text

    lowered = text.lower()
    positions = [lowered.find(term) for term in significant_terms(query)]
    positions = [pos for pos in positions if pos >= 0]
    center = min(positions) if positions else 0
    start = max(0, center - max_chars // 3)
    end = min(len(text), start + max_chars)
    passage = text[start:end]
    return f'{title}: {passage}' if title else passage

def choose_con_bert(question, index, top_k_bm25=4, evidence_top_k=5):
    global_query = ' '.join([str(question.text), *[str(opt.text) for opt in question.options]])
    global_results = retrieve(index, global_query, top_k=evidence_top_k)

    option_scores = []
    for opt in question.options:
        option_query = f'{question.text} {opt.text}'
        candidates = retrieve(index, option_query, top_k=top_k_bm25)

        if candidates:
            pairs = [(option_query, passage_for_bert(doc, option_query)) for _, doc in candidates]
            bert_scores = [
                float(score)
                for score in bert_model.predict(pairs, batch_size=16, show_progress_bar=False)
            ]
            ranked = sorted(zip(bert_scores, candidates), key=lambda row: row[0], reverse=True)
            top_scores = [score for score, _ in ranked[:3]]
            bert_score = top_scores[0] + 0.10 * float(np.mean(top_scores))
            best_doc = ranked[0][1][1]
            best_bm25_score = ranked[0][1][0]
        else:
            bert_score = float('-inf')
            best_doc = None
            best_bm25_score = 0.0

        option_scores.append({
            'option_id': int(opt.id),
            'option_text': str(opt.text),
            'score': float(bert_score),
            'evidence_score': float(bert_score),
            'best_bm25_score': float(best_bm25_score),
            'top_evidence': compact_snippet(best_doc) if best_doc else None,
        })

    option_scores.sort(key=lambda row: row['score'], reverse=True)
    best = option_scores[0]
    second_score = option_scores[1]['score'] if len(option_scores) > 1 else 0.0

    return RetrievalDecision(
        option_id=int(best['option_id']),
        option_text=str(best['option_text']),
        confidence=float(best['score'] - second_score),
        option_scores=option_scores,
        evidence=[{'score': sc, **compact_snippet(dc)} for sc, dc in global_results],
    )

In [ ]:
# Diagnostica locale: controlla che i tool matematici coprano i pattern gia implementati.
# Questa cella non chiama l'API. Serve a evitare regressioni prima di provare le nuove domande live.
class Obj:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

def q(text, options):
    return Obj(text=text, options=[Obj(id=i, text=opt) for i, opt in enumerate(options)])

TOOL_SMOKE_TESTS = [
    (
        q('The least common multiple divided by the greatest common divisor of two positive integers is 6. One integer is 4. What is the smallest possible other integer?', ['2', '6', '8', '12']),
        1,
        'lcm/gcd option check',
    ),
    (
        q('A dataset has correlation r = 0.73. If heights are converted from meters to centimeters, what is the new correlation?', ['0.0073', '0.73', '7.3', '73']),
        1,
        'correlation scale invariance',
    ),
    (
        q('If the correlation is r = 0.8, what percentage of variation is explained?', ['8%', '16%', '64%', '80%']),
        2,
        'r squared',
    ),
    (
        q('What is the degree of the field extension Q(sqrt(2)+sqrt(3)) over Q?', ['2', '3', '4', '6']),
        2,
        'field extension degree',
    ),
    (
        q('What is the order of the quotient factor group Z_4 x Z_6 / <1, 1>?', ['1', '2', '6', '24']),
        1,
        'direct product quotient order',
    ),
    (
        q('Solve for x: 2*x + 3 = 11', ['2', '3', '4', '5']),
        2,
        'basic equation',
    ),
]

for question, expected_id, label in TOOL_SMOKE_TESTS:
    decision = choose_with_agentic_tools(question, fallback=lambda _: None)
    got = None if decision is None else decision.option_id
    status = 'OK' if got == expected_id else 'FAIL'
    strategy = None if decision is None else decision.strategy
    print(f'{status:4} {label:32} expected={expected_id} got={got} strategy={strategy}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Scelta consigliata dai documenti locali: LLM istruito 1B-2B, prompt corto, nessuna spiegazione.
# Se Colab va in OOM, usa 'Qwen/Qwen2.5-0.5B-Instruct'.
LLM_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
USE_4BIT = False

model_kwargs = {}
if torch.cuda.is_available():
    model_kwargs.update({'device_map': 'auto', 'torch_dtype': torch.float16})
else:
    model_kwargs.update({'torch_dtype': torch.float32})

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model_kwargs.pop('torch_dtype', None)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, trust_remote_code=True, **model_kwargs)
llm_model.eval()
print('Loaded LLM:', LLM_MODEL_ID)
print('LLM first parameter device:', next(llm_model.parameters()).device)

In [ ]:
def format_options(question):
    return '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in question.options)

def format_evidence(evidence, max_snippets=3, max_chars=420):
    lines = []
    for i, item in enumerate(evidence[:max_snippets], 1):
        title = str(item.get('title') or item.get('id') or f'evidence {i}')
        text = re.sub(r'\s+', ' ', str(item.get('text') or '')).strip()
        if len(text) > max_chars:
            text = text[:max_chars - 3] + '...'
        lines.append(f'[{i}] {title}: {text}')
    return '\n'.join(lines) if lines else 'No retrieved evidence.'

def format_bert_ranking(bert_decision, max_options=4):
    lines = []
    for row in bert_decision.option_scores[:max_options]:
        score = row.get('score')
        score_text = 'nan' if score is None else f'{float(score):.3f}'
        lines.append(f"{row['option_id']}. {row['option_text']} (bert_score={score_text})")
    return '\n'.join(lines)

def build_llm_prompt(question, bert_decision):
    return f'''You must answer a multiple-choice quiz.
Return only the numeric option id. Do not explain.

Question:
{question.text}

Options:
{format_options(question)}

Retrieved evidence:
{format_evidence(bert_decision.evidence, max_snippets=3)}

BERT reranker ranking:
{format_bert_ranking(bert_decision)}

Answer with one option id only.'''

def generate_llm_text(prompt, max_new_tokens=8):
    messages = [
        {'role': 'system', 'content': 'You answer multiple-choice quiz questions. Return only the option id.'},
        {'role': 'user', 'content': prompt},
    ]
    if getattr(tokenizer, 'chat_template', None):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = messages[0]['content'] + '\n\n' + messages[1]['content'] + '\nAnswer:'

    inputs = tokenizer(text, return_tensors='pt')
    input_device = next(llm_model.parameters()).device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def parse_llm_choice(raw_text, question):
    valid_ids = {int(opt.id) for opt in question.options}
    for value in re.findall(r'\b\d+\b', str(raw_text)):
        option_id = int(value)
        if option_id in valid_ids:
            return option_id

    normalized_raw = ' '.join(str(raw_text).lower().split())
    for opt in question.options:
        option_text = ' '.join(str(opt.text).lower().split())
        if option_text and option_text in normalized_raw:
            return int(opt.id)

    return None

def option_text_by_id(question, option_id):
    for opt in question.options:
        if int(opt.id) == int(option_id):
            return str(opt.text)
    return ''

In [ ]:
def decision_from_tool(question, tool_decision):
    return RetrievalDecision(
        option_id=int(tool_decision.option_id),
        option_text=option_text_by_id(question, tool_decision.option_id),
        confidence=float(tool_decision.confidence),
        option_scores=[{
            'option_id': int(tool_decision.option_id),
            'option_text': option_text_by_id(question, tool_decision.option_id),
            'score': float(tool_decision.confidence),
            'strategy': tool_decision.strategy,
        }],
        evidence=[{'source': 'agentic_math_tool', 'text': tool_decision.explanation}],
    )

def choose_agentic_bert_llm(question, index, top_k_bm25=4):
    # 1) Tool deterministici: sono conservativi, quindi si possono provare su tutte le categorie.
    tool_decision = choose_with_agentic_tools(question, fallback=lambda _: None)
    if tool_decision is not None:
        return decision_from_tool(question, tool_decision)

    # 2) BM25 + BERT: produce evidenza e ranking compatto.
    bert_decision = choose_con_bert(question, index, top_k_bm25=top_k_bm25)

    # 3) LLM piccolo: riceve solo domanda, opzioni, pochi snippet e ranking BERT.
    prompt = build_llm_prompt(question, bert_decision)
    raw_answer = generate_llm_text(prompt, max_new_tokens=8)
    parsed_id = parse_llm_choice(raw_answer, question)

    if parsed_id is None:
        bert_decision.evidence.insert(0, {
            'source': 'llm_parse_fallback',
            'text': f'LLM output was not parseable: {raw_answer!r}. Falling back to BERT option.',
        })
        return bert_decision

    llm_row = {
        'option_id': int(parsed_id),
        'option_text': option_text_by_id(question, parsed_id),
        'score': 1.0,
        'strategy': 'llm_rag_with_bert_ranking',
        'raw_llm_answer': raw_answer,
    }
    option_scores = [llm_row] + [row for row in bert_decision.option_scores if int(row['option_id']) != int(parsed_id)]
    evidence = [{
        'source': 'llm',
        'model': LLM_MODEL_ID,
        'text': f'Raw LLM answer: {raw_answer}',
    }] + bert_decision.evidence

    return RetrievalDecision(
        option_id=int(parsed_id),
        option_text=option_text_by_id(question, parsed_id),
        confidence=0.50,
        option_scores=option_scores,
        evidence=evidence,
    )

In [ ]:
# Smoke test locale: controlla tool, retrieval, BERT e generazione LLM prima di chiamare l'API di gioco.
question = Obj(
    text='Who co-founded Apple Inc.?',
    options=[
        Obj(id=0, text='Steve Jobs'),
        Obj(id=1, text='Bill Gates'),
        Obj(id=2, text='Albert Einstein'),
        Obj(id=3, text='Napoleon Bonaparte'),
    ],
)

decision = choose_agentic_bert_llm(question, indexes, top_k_bm25=4)
print('Chosen:', decision.option_id, decision.option_text)
print('Top scores:', [(row.get('option_id'), row.get('option_text'), row.get('score')) for row in decision.option_scores[:4]])
print('Top evidence:', decision.evidence[0])

In [ ]:
# Parti con 1 tentativo per verificare tempi/output. Poi aumenta.
ATTEMPTS_PER_COMPETITION = 1
STRATEGY_NAME = 'bm25_bert_llm_qwen15_agentic_tools_colab'

rows = run_all_competitions(
    client=client,
    index=indexes,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
    choose_fn=lambda question, index: choose_agentic_bert_llm(question, index, top_k_bm25=4),
)

output_path = PROJECT_ROOT / 'logs' / 'simplewiki_kelm_bm25_bert_llm_agentic_tools_colab.csv'
write_logs(rows, output_path)
print(f'Wrote {len(rows)} rows to {output_path}')

In [ ]:
summarize(rows)